# TTF Natural Gas Options — Interactive Jupyter Dashboard

**Section 1 — Pricer**

This notebook replaces the Streamlit dashboard with an interactive
Jupyter interface built on `ipywidgets`. Pricing functions are imported
directly from `black76_ttf.py` placed in the **same folder** as this
notebook.

**Requirements :**
```
pip install ipywidgets numpy scipy pandas
# JupyterLab ≥ 3.0 : widgets work out of the box
# Classic Notebook  : jupyter nbextension enable --py widgetsnbextension
```

Run the two code cells below ; the price card will update live as you
move the sliders.


In [ ]:
import math
import os
import sys
from pathlib import Path

# Make TTF modules importable no matter where the notebook is launched from
_HERE = Path(globals().get("__vsc_ipynb_file__", os.getcwd())).resolve().parent
if str(_HERE) not in sys.path:
    sys.path.append(str(_HERE))
# Fallback : also append the current working directory
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

import ipywidgets as w
import pandas as pd
from IPython.display import display, HTML, clear_output

from black76_ttf import (
    b76_call, b76_put, b76_greeks,
    bach_call, bach_put, bach_greeks,
    b76_implied_vol, bach_implied_vol,
)

print("Imports OK — ready to build the Pricer UI.")


## Section 1 — Pricer

Move the sliders on the left ; the right panel refreshes with the
Call / Put prices for the selected model and the full table of Greeks.
The bottom table compares Black-76 and Bachelier side by side.


In [ ]:
# ── Widgets ────────────────────────────────────────────────────────────
F_w      = w.FloatSlider(value=30.0,  min=-20.0, max=250.0, step=0.5,
                         description="Forward F",
                         style={"description_width": "110px"},
                         layout=w.Layout(width="380px"),
                         readout_format=".1f")
K_w      = w.FloatSlider(value=30.0,  min=0.0,   max=250.0, step=0.5,
                         description="Strike K",
                         style={"description_width": "110px"},
                         layout=w.Layout(width="380px"),
                         readout_format=".1f")
Tdays_w  = w.IntSlider  (value=90,    min=1,     max=1825,  step=1,
                         description="T (days)",
                         style={"description_width": "110px"},
                         layout=w.Layout(width="380px"))
rpct_w   = w.FloatSlider(value=2.0,   min=0.0,   max=10.0,  step=0.05,
                         description="Rate (%)",
                         style={"description_width": "110px"},
                         layout=w.Layout(width="380px"),
                         readout_format=".2f")
sigma_w  = w.FloatSlider(value=50.0,  min=1.0,   max=250.0, step=1.0,
                         description="σ B76 (%)",
                         style={"description_width": "110px"},
                         layout=w.Layout(width="380px"),
                         readout_format=".0f")
sign_w   = w.FloatSlider(value=8.0,   min=0.1,   max=50.0,  step=0.1,
                         description="σₙ Bach (€/MWh)",
                         style={"description_width": "110px"},
                         layout=w.Layout(width="380px"),
                         readout_format=".1f")
type_w   = w.RadioButtons(options=["call", "put"], value="call",
                          description="Type",
                          style={"description_width": "110px"},
                          layout=w.Layout(width="280px"))
model_w  = w.Dropdown    (options=["Black-76", "Bachelier"], value="Black-76",
                          description="Model",
                          style={"description_width": "110px"},
                          layout=w.Layout(width="280px"))

out = w.Output()


# ── Pricing + HTML rendering ───────────────────────────────────────────
_CSS = """
<style>
  .ttf-card     { font-family: -apple-system, Segoe UI, Roboto, sans-serif;
                  color:#e2e8f0; background:#161625;
                  border:1px solid #2a2a42; border-radius:10px;
                  padding:14px 18px; margin:4px 0; }
  .ttf-card h3  { margin:0 0 8px 0; color:#f1f5f9; font-size:14pt;
                  border-bottom:1px solid #3b82f6; padding-bottom:4px; }
  .ttf-price    { display:flex; gap:20px; margin:8px 0 14px 0; }
  .ttf-px       { flex:1; background:#0f0f1a; border:1px solid #2a2a42;
                  border-radius:8px; padding:10px 14px; }
  .ttf-px .lbl  { color:#94a3b8; font-size:9pt; text-transform:uppercase;
                  letter-spacing:0.05em; }
  .ttf-px .val  { color:#f1f5f9; font-size:16pt; font-weight:600;
                  margin-top:2px; }
  .ttf-greeks   { border-collapse:collapse; width:100%; font-size:10pt; }
  .ttf-greeks th, .ttf-greeks td
                { padding:6px 10px; border-bottom:1px solid #2a2a42;
                  text-align:right; }
  .ttf-greeks th{ color:#94a3b8; text-align:left; font-weight:500;
                  background:#0f0f1a; }
  .ttf-greeks tr:nth-child(even) td { background:#12121d; }
  .ttf-tag      { display:inline-block; padding:2px 8px; margin-left:6px;
                  font-size:9pt; border-radius:10px; background:#1d4ed8; color:#fff; }
  .ttf-tag.bach { background:#7c3aed; }
</style>
"""


def render_pricer(F, K, Tdays, rpct, sigma_pct, sigma_n, option_type, model):
    """Compute prices/Greeks for both models and render an HTML card."""
    T     = Tdays / 365.0
    r     = rpct / 100.0
    sigma = sigma_pct / 100.0

    # Both models
    b76_c = b76_call(F, K, T, r, sigma)
    b76_p = b76_put (F, K, T, r, sigma)
    bc    = bach_call(F, K, T, r, sigma_n)
    bp    = bach_put (F, K, T, r, sigma_n)
    g76   = b76_greeks (F, K, T, r, sigma,  option_type)
    gbch  = bach_greeks(F, K, T, r, sigma_n, option_type)

    # Primary model (selected)
    if model == "Black-76":
        primary_c, primary_p, gp = b76_c, b76_p, g76
        tag_class = ""
    else:
        primary_c, primary_p, gp = bc, bp, gbch
        tag_class = "bach"

    # Put-call parity check
    df_disc = math.exp(-r * T)
    pcp_rhs = df_disc * (F - K)
    pcp_b76 = b76_c - b76_p
    pcp_bch = bc - bp

    moneyness = ("ATM" if abs(F - K) / max(F, 1e-6) < 0.005
                 else ("ITM" if (F > K) == (option_type == "call") else "OTM"))

    html = _CSS + f"""
    <div class="ttf-card">
      <h3>{model} — {option_type.upper()}
          <span class="ttf-tag {tag_class}">{model}</span>
          <span class="ttf-tag" style="background:#475569;">{moneyness}</span>
      </h3>
      <div class="ttf-price">
        <div class="ttf-px">
          <div class="lbl">Call price</div>
          <div class="val">{primary_c:.4f} €/MWh</div>
        </div>
        <div class="ttf-px">
          <div class="lbl">Put price</div>
          <div class="val">{primary_p:.4f} €/MWh</div>
        </div>
        <div class="ttf-px">
          <div class="lbl">Intrinsic ({option_type})</div>
          <div class="val">{(max(F-K,0) if option_type=='call' else max(K-F,0)):.4f}</div>
        </div>
      </div>

      <table class="ttf-greeks">
        <tr><th>Greek</th><th>Value</th><th>Unit</th></tr>
        <tr><td>Δ Delta</td>     <td>{gp.delta:+.4f}</td> <td>per unit F</td></tr>
        <tr><td>Γ Gamma</td>     <td>{gp.gamma:.6f}</td>   <td>per (€/MWh)²</td></tr>
        <tr><td>ν Vega</td>      <td>{gp.vega:.4f}</td>    <td>per unit vol (÷100 for 1 vol pt)</td></tr>
        <tr><td>Θ Theta/day</td> <td>{gp.theta:+.4f}</td>  <td>€/MWh / day</td></tr>
        <tr><td>ρ Rho</td>       <td>{gp.rho:+.6f}</td>    <td>per 1 pp change in r</td></tr>
        <tr><td>Vanna</td>       <td>{gp.vanna:+.6f}</td>  <td>∂Δ/∂σ = ∂ν/∂F</td></tr>
        <tr><td>Volga</td>       <td>{gp.volga:+.6f}</td>  <td>∂²V/∂σ²</td></tr>
      </table>
    </div>

    <div class="ttf-card">
      <h3>Side-by-side — Black-76 vs Bachelier</h3>
      <table class="ttf-greeks">
        <tr><th>Side</th><th>Black-76 price</th><th>Bachelier price</th>
            <th>Δ B76</th><th>Δ Bach</th>
            <th>ν B76</th><th>ν Bach</th></tr>
        <tr><td>Call</td>
            <td>{b76_c:.4f}</td><td>{bc:.4f}</td>
            <td>{g76.delta:+.4f}</td><td>{gbch.delta:+.4f}</td>
            <td>{g76.vega:.4f}</td><td>{gbch.vega:.4f}</td></tr>
        <tr><td>Put</td>
            <td>{b76_p:.4f}</td><td>{bp:.4f}</td>
            <td>{b76_greeks(F,K,T,r,sigma,'put').delta:+.4f}</td>
            <td>{bach_greeks(F,K,T,r,sigma_n,'put').delta:+.4f}</td>
            <td>{g76.vega:.4f}</td><td>{gbch.vega:.4f}</td></tr>
      </table>
      <p style="color:#94a3b8; font-size:9pt; margin-top:10px;">
        Put-call parity check &nbsp;·&nbsp;
        e<sup>-rT</sup>(F-K) = <b>{pcp_rhs:.6f}</b>
        &nbsp;·&nbsp; B76 C-P = {pcp_b76:.6f}
        &nbsp;·&nbsp; Bach C-P = {pcp_bch:.6f}
      </p>
    </div>
    """
    with out:
        clear_output(wait=True)
        display(HTML(html))


# ── Wire widgets → pricing ─────────────────────────────────────────────
widget_dict = {
    "F": F_w, "K": K_w, "Tdays": Tdays_w, "rpct": rpct_w,
    "sigma_pct": sigma_w, "sigma_n": sign_w,
    "option_type": type_w, "model": model_w,
}
_ = w.interactive_output(render_pricer, widget_dict)

# ── Layout ─────────────────────────────────────────────────────────────
left_col  = w.VBox([F_w, K_w, Tdays_w, rpct_w, sigma_w, sign_w])
right_col = w.VBox([type_w, model_w])
controls  = w.HBox([left_col, right_col])
display(controls, out)


---

**Tips :**

* `Black-76` is the market standard for TTF when F > ~5 €/MWh.
* `Bachelier` keeps working when F can become very low or even negative
  — useful for crisis scenarios. σₙ is the normal vol in EUR/MWh.
* The *Side-by-side* table shows both models at once so you can see the
  price gap widen when F moves away from σₙ / σ equivalence
  (rule of thumb : σₙ ≈ F · σ for ATM).

Next sections (Structures, Vol Surface, Spread TTF/HH, Expiry Dates)
can be added following the same pattern — ask to extend this notebook.
